# Layer 2: Place Name → IANA Zone

Input:  a string the user typed (city, country, etc.)
Output: the most likely IANA zone, or a list of candidates

Data source: GeoNames cities15000.txt (TSV)
Strategy: load into a dict, exact-match first, then prefix-match.

In [37]:
filename = "geonames/cities5000.txt"

In [38]:
with open(filename, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(repr(line[:300]))
        print("---")

"3039163\tSant Julià de Lòria\tSant Julia de Loria\tSan Julia,San Julià,Sant Julia de Loria,Sant Julià de Lòria,Sant-Zhulia-de-Lorija,Sant-Zhulija-de-Lorija,sainta juli'a di lori'a,saniti ጁliya de loriya,sant jwlya dy lwrya,sant jwlya ڈy lwrya,santa juliya di loriya,sheng hu li ya-de luo li ya,Сант-Жули"
---
'3039678\tOrdino\tOrdino\tOrdino,ao er di nuo,ardino,ardynw,arڈynw,awrdynw,oradino,ordino,orudino jiao qu,ʿoridino,Ордино,آردینو,آرڈینو,أوردينو,اوردینو,ऑर्डिनो,অর্ডিনো,ਓਰਡੀਨੋ,ኦርዲኖ,オルディノ教区,奥尔迪诺\t42.55623\t1.53319\tP\tPPLA\tAD\t\t05\t\t\t\t3066\t\t1296\tEurope/Andorra\t2026-04-13\n'
---
"3040051\tles Escaldes\tles Escaldes\tEhskal'des-Ehndzhordani,Ehskal'des-Ehngordani,Escaldes,Escaldes-Engordany,Les Escaldes,aisakaladesa-inagoradani,askalds-angwrdany,askalds-anګwrdany,askaldys-anjwrdany,askalڈs-angwrڈany,esakaladesa-enagordani,eskaladesa-enagordani,esukarudesu=engorudani jiao qu,lai s"
---


## Column indices we care about (from the readme)

0  geonameid
1  name
2  asciiname
3  alternatenames     (comma-separated, can be huge)
4  latitude
5  longitude
6  feature class
7  feature code
8  country code
9  cc2
10 admin1 code
11 admin2 code
12 admin3 code
13 admin4 code
14 population
15 elevation
16 dem
17 timezone            <- the magic column
18 modification date

In [39]:
with open(filename, encoding="utf-8") as f:
    for line in f:
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 18:
            continue
        name = fields[1]
        ascii_name = fields[2]
        lat = float(fields[4])
        lon = float(fields[5])
        country = fields[8]
        population = int(fields[14]) if fields[14] else 0
        zone = fields[17]
        print(f"{name} ({country}, pop {population}) -> {zone}")
        break

Sant Julià de Lòria (AD, pop 8022) -> Europe/Andorra


In [40]:
from collections import defaultdict

def load_places(path):
    """Return a dict: lowercase name -> list of place dicts."""
    index = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 18:
                continue
            try:
                place = {
                    "name": fields[1],
                    "ascii_name": fields[2],
                    "lat": float(fields[4]),
                    "lon": float(fields[5]),
                    "country": fields[8],
                    "population": int(fields[14]) if fields[14] else 0,
                    "zone": fields[17] or None,
                }
            except (ValueError, IndexError):
                continue

            # Index by both the local name and the ASCII name
            for key in {place["name"].lower(), place["ascii_name"].lower()}:
                index[key].append(place)
    return index

places = load_places(filename)
print(f"loaded {sum(len(v) for v in places.values())} place entries")
print(f"unique keys: {len(places)}")

loaded 84234 place entries
unique keys: 77744


In [41]:
def lookup(name):
    """Return matches, sorted by population descending."""
    matches = places.get(name.lower(), [])
    return sorted(matches, key=lambda p: p["population"], reverse=True)

for query in ["Paris", "Springfield", "Kolkata", "Calcutta", "Mumbai", "Tokyo"]:
    results = lookup(query)
    print(f"\n{query!r} -> {len(results)} match(es)")
    for p in results[:3]:
        print(f"    {p['name']}, {p['country']} (pop {p['population']}) -> {p['zone']}")


'Paris' -> 7 match(es)
    Paris, FR (pop 2138551) -> Europe/Paris
    Paris, US (pop 24782) -> America/Chicago
    Paris, CA (pop 12310) -> America/Toronto

'Springfield' -> 12 match(es)
    Springfield, US (pop 170188) -> America/Chicago
    Springfield, US (pop 154341) -> America/New_York
    Springfield, US (pop 114394) -> America/Chicago

'Kolkata' -> 1 match(es)
    Kolkata, IN (pop 4631392) -> Asia/Kolkata

'Calcutta' -> 1 match(es)
    Calcutta, ZA (pop 35864) -> Africa/Johannesburg

'Mumbai' -> 1 match(es)
    Mumbai, IN (pop 12691836) -> Asia/Kolkata

'Tokyo' -> 1 match(es)
    Tokyo, JP (pop 9733276) -> Asia/Tokyo


In [42]:
def top_zone(query):
    """Return the IANA zone for the most populous place matching the query."""
    results = lookup(query)
    if not results:
        return None
    return results[0]["zone"]

for q in ["Paris", "Tokyo", "Springfield", "Mumbai", "New York"]:
    print(f"{q:<12} -> {top_zone(q)}")

Paris        -> Europe/Paris
Tokyo        -> Asia/Tokyo
Springfield  -> America/Chicago
Mumbai       -> Asia/Kolkata
New York     -> None


## What we have

- A dict mapping name -> list of places
- lookup(name) -> all matches, sorted by population
- top_zone(name) -> the most likely IANA zone

## What we DON'T have

- Country filtering ("Paris, FR" vs "Paris, TX")
- Alternate names ("Bombay" -> Mumbai, "Calcutta" -> Kolkata)
- Prefix search for autocomplete
- Fallback to Layer 2b when a place has no zone

Those come later, or in Layer 1/5.

In [ ]:
from zoneinfo import ZoneInfo
from zoneinfo import ZoneInfoNotFoundError
from datetime import datetime, timezone

def is_valid_zone(name):
    if not name:
        return False
    try:
        ZoneInfo(name)
        return True
    except (ZoneInfoNotFoundError, ValueError, OSError):
        return False

def snapshot(zone_name):
    tz = ZoneInfo(zone_name)
    local = datetime.now(timezone.utc).astimezone(tz)
    return local.strftime("%H:%M %Z (%z)")

# Full pipeline: name -> zone -> local time
for q in ["Paris", "Tokyo", "Mumbai", "New York", "Springfield"]:
    z = top_zone(q)
    if z and is_valid_zone(z):
        print(f"{q:<12} -> {z:<20} {snapshot(z)}")
    else:
        print(f"{q:<12} -> NO ZONE")

Paris        -> Europe/Paris         10:17 CEST (+0200)
Tokyo        -> Asia/Tokyo           17:17 JST (+0900)
Mumbai       -> Asia/Kolkata         13:47 IST (+0530)
New York     -> NO ZONE
Springfield  -> America/Chicago      03:17 CDT (-0500)
